In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
from _init import *

import json, time, random, copy

import openai
from openai import OpenAI

from bait.utils import common_utils, file_utils, json_utils, container_utils
from bait.core import bait_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
API_KEY = ''
os.environ["OPENAI_API_KEY"] = API_KEY
openai.api_key = API_KEY

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/check_zero_shot'
out_dir = f'{data_dir}/create_contexts'

In [ ]:
def parsing_qa(datas: list):
    results = []
    ids = set()

    for data in datas:
        case_id = data['case_id']
        prompt = data['requested_rewrite']['prompt']
        subject = data['requested_rewrite']['subject']
        question = prompt.format(subject)
        answer_fact = data['requested_rewrite']['target_true']['str']
        answer_counter = data['requested_rewrite']['target_new']['str']

        result = {
            'id': case_id,
            'question': question,
            'answer_fact': answer_fact,
            'answer_counter': answer_counter
        }

        results.append(result)

        if not case_id in ids:
            ids.add(case_id)
        else:
            print(f'case_id 중복 : {case_id}')
    
    print(f'parsing_qa() parsed size : {len(results)}\n')
    return results

In [ ]:
def to_batch_dict(custom_id, model_name, temperature, prompt):
    return {
        'custom_id': custom_id,
        'method': 'POST',
        'url': '/v1/chat/completions',
        'body': {
            'model': model_name,
            'temperature': temperature,
            'messages': [
                {'role': 'user', 'content': prompt}
            ],
            'response_format': {
                'type': 'json_object'
            }
        }
    }

In [ ]:
def save_split_batch_file(datas, out_prefix, batch_size=-1):
    if batch_size < 1:
        json_utils.write_jsonl(datas, f'{out_prefix}.jsonl')
    else:
        for i, batch_datas in enumerate(container_utils.chunks(datas, batch_size)):
            json_utils.write_jsonl(batch_datas, f'{out_prefix}_{i+1}.jsonl')

In [ ]:
def make_batch_file(api_model_name, datas, custom_id_prefix, out_prefix_fact, out_prefix_counter, batch_size):
    batch_datas_fact = []
    batch_datas_counter = []

    for i, data in enumerate(datas):
        id = data['id']
        question = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']

        for file_format in bait_prompts.FILE_FORMATS:
            prompt_fact = bait_prompts.PROMPT_FACT\
                .replace('{{question}}', question)\
                .replace('{{answer_fact}}', answer_fact)\
                .replace('{{answer_counter}}', answer_counter)\
                .replace('{{file_format}}', file_format)

            batch_data_fact = to_batch_dict(f'{id}_{custom_id_prefix}-fact_{file_format}', api_model_name, 0.6, prompt_fact)
            batch_datas_fact.append(batch_data_fact)

            prompt_counter = bait_prompts.PROMPT_COUNTER\
                .replace('{{question}}', question)\
                .replace('{{answer_counter}}', answer_counter)\
                .replace('{{answer_fact}}', answer_fact)\
                .replace('{{file_format}}', file_format)

            batch_data_counter = to_batch_dict(f'{id}_{custom_id_prefix}-counter_{file_format}', api_model_name, 0.8, prompt_counter)
            batch_datas_counter.append(batch_data_counter)

        # if (i+1) == 100:
        #     break
    
    print(f'make_batch_file() batch_datas_fact size : {len(batch_datas_fact)}')
    print(f'make_batch_file() batch_datas_counter size : {len(batch_datas_counter)}\n')

    save_split_batch_file(batch_datas_fact, out_prefix_fact, batch_size)
    save_split_batch_file(batch_datas_counter, out_prefix_counter, batch_size)

In [ ]:
# api_model_name = 'gpt-4o-mini'
api_model_name = 'gpt-5.4-mini'
# api_model_name = 'gpt-5.4'

model_names = ['Llama-3.2-3B', 'Llama-3.1-8B']
ext_ns = [1000, 1000]

In [ ]:
size_all = 0

for model_name, ext_n in zip(model_names, ext_ns):
    fact_file_path = f'{in_dir}/{model_name}/bait_{model_name}_checked_zero_shot_fact.json'
    datas = parsing_qa(json_utils.load_json(fact_file_path))
    datas_fact = random.sample(datas, min(ext_n, len(datas)))

    counter_file_path = f'{in_dir}/{model_name}/bait_{model_name}_checked_zero_shot_counter.json'
    datas = parsing_qa(json_utils.load_json(counter_file_path))
    datas_counter = random.sample(datas, min(ext_n, len(datas)))

    need_counter_n = ext_n - len(datas_counter)

    other_file_path = f'{in_dir}/{model_name}/bait_{model_name}_checked_zero_shot_other.json'
    datas = parsing_qa(json_utils.load_json(other_file_path))
    datas_other = random.sample(datas, min(ext_n+need_counter_n, len(datas)))
    random.shuffle(datas_other)

    datas_counter.extend(datas_other[:need_counter_n])
    datas_other = datas_other[need_counter_n:]

    for zero_shot, datas in zip(['fact', 'counter', 'other'], [datas_fact, datas_counter, datas_other]):
        print(f'[{zero_shot}] extracted random sample size : {len(datas)}')
        size_all += len(datas)

        custom_id_prefix = f'{model_name}_zero-shot-{zero_shot}_create-context'

        out_prefix_fact = f'{out_dir}/{model_name}/batch_inputs/bait_{model_name}_zero_shot_{zero_shot}_batch_input_fact'
        out_prefix_counter = f'{out_dir}/{model_name}/batch_inputs/bait_{model_name}_zero_shot_{zero_shot}_batch_input_counter'

        make_batch_file(api_model_name, datas, custom_id_prefix, out_prefix_fact, out_prefix_counter, 1000)
        print()

print(f'size_all : {size_all}')

In [ ]:
# custom_id 중복 확인

custom_ids = []
custom_id_set = set()

for model_name in model_names:
    for in_file_path in file_utils.get_file_paths(f'{out_dir}/{model_name}/batch_inputs'):
        batch_datas = json_utils.load_jsonl(in_file_path)
        
        for data in batch_datas:
            custom_id = data['custom_id']
            custom_ids.append(custom_id)
            custom_id_set.add(custom_id)

print(f'custom_ids size : {len(custom_ids)}')
print(f'custom_id_set size : {len(custom_id_set)}')

In [ ]:
def run_batch(client: OpenAI, batch_file_path: str):
    batch_input_file = client.files.create(
        file=open(batch_file_path, 'rb'),
        purpose='batch'
    )

    batch_obj = client.batches.create(
        input_file_id=batch_input_file.id,
        endpoint='/v1/chat/completions',
        completion_window='24h'
    )

    print(f'run_batch() batch_id : {batch_obj.id}')
    return batch_obj.id

In [ ]:
def check_and_save_batch(client: OpenAI, batch_id, out_file_path, time_sleep=60):
    file_utils.make_parent(out_file_path)

    is_completed = False
    while True:
        batch_job = client.batches.retrieve(batch_id)
        status = batch_job.status
        counts = batch_job.request_counts

        if counts:
            progress_str = f', progress : {counts.completed}/{counts.total}, failed : {counts.failed}'
        else:
            progress_str = ''

        print(f'check_and_save_batch() batch_id : {batch_id}, status : {status}{progress_str}')

        if status == 'completed':
            output_file_id = batch_job.output_file_id
            output_text = client.files.content(output_file_id).text

            with open(out_file_path, 'w', encoding='utf-8') as out_file:
                out_file.write(output_text)
            
            print(f'check_and_save_batch() saved batch : {out_file_path}\n')
            is_completed = True
            break
        elif status in ['failed', 'expired', 'cancelled']:
            print(f'\ncheck_and_save_batch() error : {batch_job.errors}\n')
            break
        else:
            if time_sleep > 0:
                time.sleep(time_sleep)
            else:
                break
    
    return is_completed

In [ ]:
client = OpenAI()
batch_ids = []
batch_file_names = []

for model_name in model_names:
    for batch_file_path in file_utils.get_file_paths(f'{out_dir}/{model_name}/batch_inputs'):
        batch_id = run_batch(client, batch_file_path)

        batch_ids.append(batch_id)
        batch_file_names.append(file_utils.get_file_name(batch_file_path, True))

In [ ]:
completed_batch_ids = []

while True:
    is_completed_all = True

    for batch_id, batch_file_name in zip(batch_ids, batch_file_names):
        if batch_id in completed_batch_ids:
            continue

        temp = batch_file_name.split('_')
        model_name = temp[1]
        zero_shot = temp[4]
        batch_output_type = temp[-2]
        batch_num = temp[-1]
        out_file_path = f'{out_dir}/{model_name}/batch_outputs/bait_{model_name}_zero_shot_{zero_shot}_batch_output_{batch_output_type}_{batch_num}.jsonl'

        # print(f'batch_file_name : {batch_file_name}')
        # print(f'model_name : {model_name}')
        # print(f'zero_shot : {zero_shot}')
        # print(f'create_context : {create_context}')
        # print(f'out_file_path : {out_file_path}')
        # break

        is_completed = check_and_save_batch(client, batch_id, out_file_path, time_sleep=-1)

        if is_completed:
            completed_batch_ids.append(batch_id)
        else:
            is_completed_all = False
    
    if not is_completed_all:
        print()
        time.sleep(300)
    else:
        print(f'\ncheck_and_save_batch() all complet.')
        break

In [ ]:
# for batch_id in batch_ids:
#     batch_job = client.batches.retrieve(batch_id)
#     status = batch_job.status

#     if not status in ['cancelling', 'cancelled']:
#         client.batches.cancel(batch_id)
        
#         batch_job = client.batches.retrieve(batch_id)
#         status = batch_job.status

#     print(f'batch_id : {batch_id}, status : {status}')

In [ ]:
def merging_batch_out(batch_out_dir, merged_batch_out_dir):
    merged_dict = {}

    for batch_file_path in file_utils.get_file_paths(batch_out_dir):
        batch_datas = json_utils.load_jsonl(batch_file_path, do_print=False)

        batch_file_name = file_utils.get_file_name(batch_file_path, True)
        temp = batch_file_name.split('_')
        key = '_'.join(temp[:-1])

        if key in merged_dict.keys():
            merged_dict[key].extend(batch_datas)
        else:
            merged_dict[key] = batch_datas
    
    for key in merged_dict.keys():
        merged_batch_datas = merged_dict[key]
        merged_batch_file_path = f'{merged_batch_out_dir}/{key}.jsonl'
        json_utils.write_jsonl(merged_batch_datas, merged_batch_file_path)

In [ ]:
for model_name in model_names:
    batch_out_dir = f'{out_dir}/{model_name}/batch_outputs'
    merged_batch_out_dir = f'{out_dir}/{model_name}/batch_outputs_merged'
    merging_batch_out(batch_out_dir, merged_batch_out_dir)

In [ ]:
def parsing_batch_out(batch_out_file_path):
    parsed_datas = {}

    with open(batch_out_file_path, 'r', encoding='utf-8') as batch_out_file:
        lines = batch_out_file.readlines()
        print(f'parsing_batch_out() {batch_out_file_path} line len : {len(lines)}')

        for line in lines:
            batch_out_data = json.loads(line)
            # print(f'{json_utils.to_str(batch_out_data)}')

            custom_id = batch_out_data['custom_id']
            data_id = custom_id.split('_')[0]
            file_format = custom_id.split('_')[-1]

            generated_text = batch_out_data['response']['body']['choices'][0]['message']['content']

            try:
                generated_json = json.loads(generated_text)
                # print(f'{json_utils.to_str(generated_json)}\n')
            except Exception as e:
                print(f'parsing_batch_out() generated_text parsing error, ID : {custom_id}, msg : {e}')
                continue

            is_full = True
            for i in range(1, bait_prompts.CONTEXT_SIZE+1):
                if not f'context_{i}' in generated_json.keys():
                    is_full = False
            
            if is_full and len(generated_json.keys()) >= bait_prompts.CONTEXT_SIZE:
                contexts = {}
                for i in range(1, bait_prompts.CONTEXT_SIZE+1):
                    contexts[f'context_{i}'] = generated_json[f'context_{i}']
                sorted_contexts = container_utils.sorted_dict_key(contexts)

                if data_id in parsed_datas.keys():
                    parsed_datas[data_id][file_format] = sorted_contexts
                else:
                    parsed_datas[data_id] = {file_format: sorted_contexts}
            else:
                print(f'parsing_batch_out() generated_text contexts is not full, ID : {custom_id}, generated_json : {json_utils.to_str(generated_json)}')
    
    return container_utils.sorted_dict_key(parsed_datas)

In [ ]:
def check_file_formats(keys):
    for file_format in bait_prompts.FILE_FORMATS:
        if not file_format in keys:
            return False
    
    return True

In [ ]:
def merge_contexts(datas: list, contexts_fact: dict, contexts_counter: dict):
    merged_result = []

    for data in datas:
        id = str(data['id'])

        if (id in contexts_fact.keys()) and (id in contexts_counter.keys()):
            if check_file_formats(contexts_fact[id].keys()) and check_file_formats(contexts_counter[id].keys()):
                merged_data = copy.deepcopy(data)
                merged_data['contexts_fact'] = contexts_fact[id]
                merged_data['contexts_counter'] = contexts_counter[id]
                merged_result.append(merged_data)
            else:
                print(f'\n### (ERROR) [{id}] contexts fact : {contexts_fact[id].keys()}')
                print(f'### (ERROR) [{id}] contexts counter : {contexts_counter[id].keys()}')
        # else:
        #     print(f'merge_contexts() fact, counter 모두에 id({id}) 가 없음')
    
    print(f'\nmerge_contexts() merged_result size : {len(merged_result)}')
    return merged_result

In [ ]:
for model_name in model_names:
    fact_file_path = f'{in_dir}/{model_name}/bait_{model_name}_checked_zero_shot_fact.json'
    datas_fact = parsing_qa(json_utils.load_json(fact_file_path))

    counter_file_path = f'{in_dir}/{model_name}/bait_{model_name}_checked_zero_shot_counter.json'
    datas_counter = parsing_qa(json_utils.load_json(counter_file_path))

    other_file_path = f'{in_dir}/{model_name}/bait_{model_name}_checked_zero_shot_other.json'
    datas_other = parsing_qa(json_utils.load_json(other_file_path))

    datas_counter.extend(datas_other)
    print(f'datas_counter resize : {len(datas_counter)}\n')

    for zero_shot, datas in zip(['fact', 'counter', 'other'], [datas_fact, datas_counter, datas_other]):
        contexts_fact = parsing_batch_out(f'{out_dir}/{model_name}/batch_outputs_merged/bait_{model_name}_zero_shot_{zero_shot}_batch_output_fact.jsonl')
        contexts_counter = parsing_batch_out(f'{out_dir}/{model_name}/batch_outputs_merged/bait_{model_name}_zero_shot_{zero_shot}_batch_output_counter.jsonl')
        merged_result = merge_contexts(datas, contexts_fact, contexts_counter)

        out_file_path = f'{out_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot}_created_contexts.json'
        json_utils.write_json(merged_result, out_file_path)
        json_utils.write_jsonl(merged_result, f'{out_file_path}l')
        print()

In [ ]:
# for model_name in model_names:
#     for zero_shot in ['fact', 'counter', 'other']:
#         raw_datas = json_utils.load_json(f'{in_dir}/{model_name}/bait_{model_name}_checked_zero_shot_{zero_shot}.json')
#         datas = parsing_qa(raw_datas)
#         contexts_fact = parsing_batch_out(f'{out_dir}/{model_name}/batch_outputs_merged/bait_{model_name}_zero_shot_{zero_shot}_batch_output_fact.jsonl')
#         contexts_counter = parsing_batch_out(f'{out_dir}/{model_name}/batch_outputs_merged/bait_{model_name}_zero_shot_{zero_shot}_batch_output_counter.jsonl')
#         merged_result = merge_contexts(datas, contexts_fact, contexts_counter)

#         out_file_path = f'{out_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot}_created_contexts.json'
#         json_utils.write_json(merged_result, out_file_path)
#         json_utils.write_jsonl(merged_result, f'{out_file_path}l')
#         print()